# M1 · Student A — HPatches pipeline + SIFT/ORB baseline numbers
**CSE 681 · Project 26 — Compact Deep Feature Descriptors vs. Classical SIFT/ORB**

| | |
|---|---|
| **Milestone** | M1 — Proposal & Literature Review (Week 3) |
| **Owner** | Student A (Data & Baseline Lead) |
| **Runtime** | CPU is fine · about 15 min |

**This notebook produces (saved to the shared Drive folder `DIP_P26/`):**
- `tables/m1_dataset.tex`: dataset summary (report §3, *Dataset & Baseline Plan*)
- `tables/m1_baseline.tex`: **SIFT vs ORB baseline numbers**. The catalog requires these at M1.
- `figures/m1_hpatches_examples.png`, `figures/m1_baseline_mma.pdf`, `figures/m1_baseline_matches.png`
- `results/m1_baseline_pairs.csv`: raw per-pair results (Student C reuses them)

> First time? Read `README.md`: one teammate creates the shared folder `DIP_P26` on Google Drive,
> uploads `p26lib.py` into it and shares it; everyone adds a shortcut to *My Drive*.

In [ ]:
#@title 0 · Setup (run every session)
import os, sys, json, time
SMOKE_TEST = False   # True = 1-2 minute dry run on tiny SYNTHETIC data (to check the code). NEVER report smoke numbers.
SMOKE_TEST = SMOKE_TEST or os.environ.get("P26_SMOKE") == "1"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJ, DATA = "/content/drive/MyDrive/DIP_P26", "/content/data"
except ImportError:                       # running locally / Kaggle
    PROJ = os.environ.get("P26_PROJ", os.path.abspath("DIP_P26"))
    DATA = os.environ.get("P26_DATA", os.path.abspath("data"))
os.makedirs(PROJ, exist_ok=True); os.makedirs(DATA, exist_ok=True)
sys.path.insert(0, PROJ)
assert os.path.exists(os.path.join(PROJ, "p26lib.py")), f"Upload p26lib.py into {PROJ} first (see README)."

import importlib, p26lib as P; importlib.reload(P)
import numpy as np, pandas as pd, cv2, torch
import matplotlib; import matplotlib.pyplot as plt
P.set_seed(P.SEED)
D = P.project_dirs(PROJ)
TAG = "SMOKE_" if SMOKE_TEST else ""          # smoke-test outputs never overwrite real ones
def OUT(kind, name):                           # e.g. OUT("tables", "m1_baseline.tex")
    return os.path.join(D[kind], TAG + name)
DEVICE = P.get_device()
print("SMOKE_TEST =", SMOKE_TEST, "| device =", DEVICE)
print(json.dumps(P.env_info(), indent=1))

In [ ]:
#@title 1 · HPatches sequences: uses the zip in Drive DIP_P26/data/ (copied + unzipped to the fast Colab disk, ~2 min per session)
if SMOKE_TEST:
    HP_ROOT = P.make_synthetic_hpatches(os.path.join(DATA, "synthetic_hpatches"), n_per_kind=2)
else:
    HP_ROOT = P.download_hpatches(DATA)
SEQS = P.list_sequences(HP_ROOT)          # D2-Net protocol: 108 sequences (52 illumination + 56 viewpoint)
print(len(SEQS), "sequences |", sum(P.seq_kind(s) == "illumination" for s in SEQS), "illumination,",
      sum(P.seq_kind(s) == "viewpoint" for s in SEQS), "viewpoint")

## 2 · Look at the data
HPatches has 116 sequences of 6 images each. Image 1 is the reference, and `H_1_i` is the ground-truth homography that maps
image 1 onto image *i*. Sequences named `i_*` change **illumination**; sequences named `v_*` change **viewpoint**. We follow the
D2-Net protocol and drop 8 very high-resolution sequences, which leaves 108.

In [ ]:
#@title 2a · Dataset statistics -> tables/m1_dataset.tex
all_seqs = P.list_sequences(HP_ROOT, exclude=[])
shapes = [cv2.imread(os.path.join(s, "1.ppm"), cv2.IMREAD_GRAYSCALE).shape for s in SEQS]
stats = pd.DataFrame([
    {"Item": "Sequences (total / used)", "Value": f"{len(all_seqs)} / {len(SEQS)}"},
    {"Item": "Illumination / viewpoint (used)", "Value": f"{sum(P.seq_kind(s)=='illumination' for s in SEQS)} / {sum(P.seq_kind(s)=='viewpoint' for s in SEQS)}"},
    {"Item": "Image pairs evaluated (1 vs 2..6)", "Value": str(5 * len(SEQS))},
    {"Item": "Image size (median HxW)", "Value": f"{int(np.median([s[0] for s in shapes]))} x {int(np.median([s[1] for s in shapes]))}"},
    {"Item": "Excluded (D2-Net protocol)", "Value": ", ".join(P.D2NET_EXCLUDED)},
    {"Item": "License", "Value": "HPatches: research use (see repo LICENSE)"},
])
display(stats)
P.df_to_latex(stats, OUT("tables", "m1_dataset.tex"), "HPatches subset used in this project.", "tab:data_stats")

In [ ]:
#@title 2b · Example sequences + ground-truth sanity check (warp image 1 onto image 6)
ex = [s for s in SEQS if P.seq_kind(s) == "illumination"][0], [s for s in SEQS if P.seq_kind(s) == "viewpoint"][0]
fig, axs = plt.subplots(2, 7, figsize=(20, 5.5))
for r, sd in enumerate(ex):
    imgs, Hs = P.load_sequence(sd)
    for i in range(6):
        axs[r, i].imshow(imgs[i], cmap="gray"); axs[r, i].set_title(f"{os.path.basename(sd)} #{i+1}", fontsize=8)
    h, w = imgs[5].shape
    warped = cv2.warpPerspective(imgs[0], Hs[6], (w, h))
    axs[r, 6].imshow(np.dstack([warped, imgs[5], warped]))   # magenta/green: aligned = gray
    axs[r, 6].set_title("GT check: warp(1) vs 6", fontsize=8)
for a in axs.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(OUT("figures", "m1_hpatches_examples.png"), dpi=150); plt.show()

## 3 · Baseline: OpenCV SIFT and ORB + mutual-nearest-neighbour matching + RANSAC
This is the Week 9 lab pipeline, run under the evaluation protocol that Student C owns (`p26lib` section 7):
- up to **2000 keypoints** per image for every method
- **mutual nearest neighbour (MNN)** matching, which is parameter-free and is the standard HPatches MMA protocol
- **MMA@t**: fraction of matches whose reprojection error under the GT homography is ≤ t px, for t = 1…10
- **H-acc@t**: RANSAC homography (3 px threshold), counted correct when its mean corner error is ≤ t px

In [ ]:
#@title 3a · Run the baseline (~5 min on Colab CPU)
N_FEATURES = 2000
baselines = [P.SIFTExtractor(N_FEATURES), P.ORBExtractor(N_FEATURES)]
df = pd.concat([P.run_benchmark(e, SEQS, match_mode="mnn") for e in baselines], ignore_index=True)
df.to_csv(OUT("results", "m1_baseline_pairs.csv"), index=False)
df.head()

In [ ]:
#@title 3b · Baseline numbers table -> tables/m1_baseline.tex  (paste into report section 3)
summ = P.summarize(df)
cols = ["mma@1", "mma@3", "mma@5", "mma@10", "hacc@1", "hacc@3", "hacc@5", "n_matches", "extract_ms", "bytes_per_desc"]
summ = summ[cols].rename(columns={"n_matches": "matches", "extract_ms": "ms/img", "bytes_per_desc": "B/desc"})
display(summ.round(3))
P.df_to_latex(summ.round(3), OUT("tables", "m1_baseline.tex"),
              "M1 baseline on HPatches (108 sequences, 540 pairs, 2000 keypoints, MNN matching). "
              "MMA = mean matching accuracy, H-acc = homography accuracy at 1/3/5 px.", "tab:m1_baseline",
              bold_max=["mma@1", "mma@3", "mma@5", "mma@10", "hacc@1", "hacc@3", "hacc@5"])
fig = P.plot_mma(df, OUT("figures", "m1_baseline_mma.pdf")); plt.show()

In [ ]:
#@title 3c · Qualitative: RANSAC inlier matches on a hard viewpoint pair
sd = [s for s in SEQS if P.seq_kind(s) == "viewpoint"][0]
imgs, Hs = P.load_sequence(sd)
fig, axs = plt.subplots(len(baselines), 1, figsize=(14, 5 * len(baselines)))
for ax, e in zip(np.atleast_1d(axs), baselines):
    H, f1, f2, m, inl = P.register(imgs[0], imgs[4], e)
    err = P.corner_error(H, Hs[5], imgs[0].shape[1], imgs[0].shape[0])
    ax.imshow(P.draw_inlier_matches(imgs[0], imgs[4], f1, f2, m, inl)[..., ::-1])
    ax.set_title(f"{e.name}: {len(m)} MNN matches, {inl.sum()} RANSAC inliers, corner error {err:.1f}px  ({os.path.basename(sd)} 1-5)")
    ax.axis("off")
plt.tight_layout(); plt.savefig(OUT("figures", "m1_baseline_matches.png"), dpi=120); plt.show()

In [ ]:
#@title 3d · Sentences you can adapt for the report
s = P.summarize(df)
for m in ["SIFT", "ORB"]:
    o, i, v = (s.loc[(m, k)] for k in ["overall", "illumination", "viewpoint"])
    print(f"{m}: MMA@3 = {o['mma@3']:.3f} (illum {i['mma@3']:.3f}, viewp {v['mma@3']:.3f}); "
          f"H-acc@3 = {o['hacc@3']:.3f}; {o['n_matches']:.0f} matches/pair; {o['extract_ms']:.0f} ms/image; "
          f"{int(o['bytes_per_desc'])} B/descriptor")

## 4 · Reproduction check: run the baseline under the SuperPoint paper's setup
Our main protocol (full resolution, 2000 keypoints, 108 sequences) differs from DeTone et al. (2018), Table 3. To show that our
baseline is implemented correctly, we **also** run SIFT/ORB under their setup: images resized to **480×640**, **1000 keypoints**,
**all 116 sequences**, OpenCV-default RANSAC (3 px, 2000 iterations, confidence 0.995). Then we compare with their published H-acc.

This run is **only** a sanity check. Every other number in the project stays on the main protocol. Small remaining differences come
from the OpenCV version and RANSAC randomness. **Check the published values against the paper's table before citing them.**

In [ ]:
#@title 4 · SIFT/ORB under the SuperPoint protocol vs. published numbers -> tables/m1_reproduction.tex
PUBLISHED_SUPERPOINT = {"SIFT": (0.424, 0.676, 0.759), "ORB": (0.150, 0.395, 0.538)}   # H-acc @ 1/3/5 px, DeTone et al. 2018, Table 3 -- VERIFY
pr = P.PROTOCOLS["superpoint"]
SEQS_ALL = P.list_sequences(HP_ROOT, exclude=pr["exclude"])
rows = []
for e in [P.SIFTExtractor(pr["n_features"]), P.ORBExtractor(pr["n_features"])]:
    g = P.run_benchmark(e, SEQS_ALL, resize=pr["resize"], ransac_thresh=pr["ransac_thresh"],
                        ransac_iters=pr["ransac_iters"], ransac_conf=pr["ransac_conf"])
    pub = PUBLISHED_SUPERPOINT[e.name]
    for t, pv in zip(P.HOMOG_THRESHOLDS, pub):
        rows.append({"method": e.name, "eps (px)": t, "ours": g[f"hacc@{t}"].mean(), "published": pv,
                     "difference": g[f"hacc@{t}"].mean() - pv})
repro = pd.DataFrame(rows)
repro.to_csv(OUT("results", "m1_reproduction.csv"), index=False)
display(repro.round(3))
P.df_to_latex(repro.round(3), OUT("tables", "m1_reproduction.tex"),
              "Reproduction check: homography accuracy of our SIFT/ORB baseline under the protocol of \\citet{detone2018superpoint} "
              "(480x640, 1000 keypoints, 116 sequences) vs. their published values.", "tab:m1_repro")

## ✅ Deliverable checklist & where it goes in the report

- [ ] `tables/m1_dataset.tex` → **§3 Dataset & Baseline Plan** (also fill the template's dataset table: HPatches · 108 seq / 540 pairs · https://github.com/hpatches/hpatches-dataset)
- [ ] `tables/m1_baseline.tex` → **§3**: write "Preliminary baseline numbers: …" with the sentences from cell 3d
- [ ] `figures/m1_baseline_mma.pdf` → optional figure in §3
- [ ] `tables/m1_reproduction.tex` → §3: *"Under the SuperPoint protocol our baseline reproduces the published H-acc within …"*. Add `detone2018superpoint` to `references.bib`.
- [ ] Commit this notebook and the CSV to the team GitHub repo
- [ ] **Literature review (A owns the references table):** at least 4 refs, at least 2 from 2023–2026, one sentence of relevance for each, all in `references.bib`